In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from typing import List, Dict, Any
from langchain.schema import Document
import sys
from tqdm import tqdm
import time

c:\Users\HP\anaconda3\envs\mchatbot\lib\site-packages\pinecone\data\index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
def load_pdf_data(pdf_path: str) -> List[Document]:
    """Load and extract text from a PDF file."""
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"The PDF file '{pdf_path}' was not found.")
    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        print(f"Loaded {len(documents)} document(s) from {pdf_path}")
        return documents
    except Exception as e:
        raise Exception(f"Failed to load PDF '{pdf_path}': {str(e)}")

In [3]:
def text_split(extracted_data: List[Document]) -> List[Document]:
    """Split extracted documents into smaller, manageable chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        is_separator_regex=False
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    print(f"Split {len(extracted_data)} page(s) into {len(text_chunks)} chunks")
    return text_chunks


In [4]:

def download_hugging_face_embeddings() -> HuggingFaceEmbeddings:
    """Download and initialize the HuggingFace sentence-transformer embeddings model."""
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )



In [5]:

def initialize_pinecone(index_name: str, api_key: str) -> Pinecone:
    """Initialize Pinecone connection and create a serverless index if it doesn't exist."""
    if not api_key:
        raise ValueError("Pinecone API key is not set. Please set it in the .env file or directly.")
    pc = Pinecone(api_key=api_key)
    if index_name not in pc.list_indexes().names():
        print(f"Pinecone index '{index_name}' not found. Creating a new one...")
        pc.create_index(
            name=index_name,
            dimension=768,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
        print(f"Successfully created new Pinecone index: '{index_name}'")
    else:
        print(f"Found existing Pinecone index: '{index_name}'")
    return pc

In [6]:
def get_prompt_template() -> str:
    """Returns the complete prompt template string."""
    SYSTEM_PROMPT = """You are MediAI, a helpful and certified medical assistant AI. Your primary goal is to provide safe, factual, and clear information based on the context provided.

Follow these rules strictly:
1.  **Medical Focus:** Only answer questions related to health, medicine, symptoms, conditions, and treatments.
2.  **Non-Medical Queries:** If the user asks a non-medical question (e.g., 'hello', 'who are you?'), you MUST respond with: "I specialize in medical topics. Please ask health-related questions about symptoms, conditions, or treatments."
3.  **No Personal Advice:** Never provide personal medical advice, diagnoses, or prescriptions. You are not a substitute for a human doctor.
4.  **Advise Consultation:** For any queries about specific symptoms, treatment plans, or serious conditions, always end your response by strongly advising the user to consult a healthcare professional.
5.  **Factual & Concise:** Base your answers strictly on the provided context. Be clear and concise. Use bullet points for lists (e.g., symptoms, prevention tips).
6.  **Safety First:** If the context is insufficient or you are unsure, state that you cannot provide a reliable answer and recommend consulting a doctor.
"""
    prompt_template = f"""
{SYSTEM_PROMPT}

CONTEXT:
{{context}}

QUESTION:
{{question}}

Medical Answer:
"""
    return prompt_template


In [7]:
from dotenv import load_dotenv
from langchain_community.vectorstores import Pinecone as PineconeVectorStore
from langchain_community.llms import CTransformers
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
import re

# --- Configuration ---
load_dotenv()
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
index_name = "medical-chatbot2"
PDF_PATH = "data/The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf"
MODEL_PATH = "model/llama-2-7b-chat.ggmlv3.q4_0.bin"


In [8]:

# --- Initialization ---
try:
    print("--- Initializing Pinecone, Embeddings, and LLM ---")
    pc = initialize_pinecone(index_name, PINECONE_API_KEY)
    embeddings = download_hugging_face_embeddings()
    llm = CTransformers(
        model=MODEL_PATH,
        model_type="llama",
        config={
            'max_new_tokens': 1024,
            'temperature': 0.2,
            'context_length': 4096,
            'gpu_layers': 50 if os.environ.get('USE_GPU', 'false').lower() == 'true' else 0,
            'threads': os.cpu_count() or 4
        }
    )
    print("All components initialized successfully.")
except Exception as e:
    print(f"[Initialization Failed] {e}")
    sys.exit(1)

--- Initializing Pinecone, Embeddings, and LLM ---
Found existing Pinecone index: 'medical-chatbot2'
All components initialized successfully.


In [9]:
print("\n--- Create and store embeddings using LangChain's Pinecone integration ---")
try:
    documents = load_pdf_data(PDF_PATH)
    text_chunks = text_split(documents)
    
    # Store embeddings in Pinecone
    PineconeVectorStore.from_documents(
        documents=text_chunks,
        embedding=embeddings,
        index_name=index_name
    )
    
    print("\nEmbeddings stored successfully.")
    index_stats = pc.Index(index_name).describe_index_stats()
    print(f"Total vectors: {index_stats.total_vector_count}")
except Exception as e:
    print(f"[Document Processing Failed] {e}")



--- Create and store embeddings using LangChain's Pinecone integration ---
Loaded 759 document(s) from data/The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf
Split 759 page(s) into 4110 chunks

Embeddings stored successfully.
Total vectors: 4110


In [10]:
print("\n--- Setting up RetrievalQA Chain ---")
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)



--- Setting up RetrievalQA Chain ---


In [11]:
PROMPT = PromptTemplate(
    template=get_prompt_template(),
    input_variables=["context", "question"]
)

In [12]:
retriever = docsearch.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        'k': 5,
        'score_threshold': 0.7,
    }
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

print("RetrievalQA chain ready.")

RetrievalQA chain ready.


In [13]:
def is_medical_query(query: str) -> bool:
    """Advanced medical query detection with regex patterns."""
    MEDICAL_KEYWORDS = ['pain', 'fever', 'headache', 'nausea', 'vomit', 'rash', 'swelling', 'bleeding', 'dizziness', 'fatigue', 'cough', 'sneeze', 'sore throat', 'shortness of breath', 'diabetes', 'cancer', 'hypertension', 'asthma', 'arthritis', 'alzheimer', 'parkinson', 'stroke', 'heart attack', 'pneumonia', 'bronchitis', 'medicine', 'medication', 'treatment', 'therapy', 'surgery', 'vaccine', 'antibiotic', 'antiviral', 'chemotherapy', 'radiation', 'heart', 'lung', 'liver', 'kidney', 'brain', 'stomach', 'intestine', 'health', 'medical', 'diagnosis', 'symptom', 'patient', 'doctor', 'hospital', 'clinic', 'pharmacy', 'prescription', 'dosage', 'side effect']
    MEDICAL_PREFIXES = ['what is', 'how to treat', 'symptoms of', 'causes of', 'diagnosis of', 'treatment for', 'medication for', 'side effects of', 'recovery from', 'prevention of', 'risk factors for', 'prognosis of', 'complications of']
    query = query.lower().strip()
    if any(re.search(rf'\b{keyword}\b', query) for keyword in MEDICAL_KEYWORDS):
        return True
    if any(query.startswith(prefix) for prefix in MEDICAL_PREFIXES):
        return True
    if re.match(r'what (causes|triggers) .+', query) or re.match(r'how to (prevent|avoid) .+', query):
        return True
    return False

def format_response(response: str) -> str:
    """Format the LLM response for better readability and add a disclaimer."""
    response = response.replace('Medical Answer:', '').strip()
    if "consult a" not in response.lower() and "medical professional" not in response.lower():
        response += "\n\n**Disclaimer:** This is for informational purposes only. Always consult a healthcare professional for medical advice."
    return response

In [14]:
# --- Run the chatbot query ---
user_query = "What are the common causes of fever?" # You can change this query
print(f"\n--- User Query: {user_query} ---")

if is_medical_query(user_query):
    result = qa_chain.invoke({"query": user_query})
    
    if result.get("result"):
        formatted_response = format_response(result["result"])
        print("\n--- MediAI Response ---")
        print(formatted_response)
    else:
        print("I couldn't generate a response. Please try rephrasing your medical question.")
else:
    print("I specialize in medical topics. Please ask health-related questions about symptoms, conditions, or treatments.")



--- User Query: What are the common causes of fever? ---

--- MediAI Response ---
The common causes of fever are infectious diseases such as pneumonia, tuberculosis, meningitis, and urinary tract infections. Autoimmune diseases, allergic reactions, and tumors can also cause fever. Additionally, medication side effects and environmental factors like heat stroke or sun exposure can lead to an elevated body temperature. It is important to consult a healthcare professional for proper diagnosis and treatment of fever.
